# Phase 6 DAEAC Diagnostics Kaggle Driver

Notebook này chỉ là driver cho workflow diagnostics. Toàn bộ logic phân tích nằm trong `ecg_thesis/src/` và `ecg_thesis/scripts/phase6_daeac_diagnostics/`.

## Cần attach trên Kaggle

1. Dataset `.npz` đã preprocess, chứa thư mục hoặc file tương đương:
   - `mitdb_ds1_daeac.npz`
   - `mitdb_ds2_daeac.npz`
   - `mitdb_ds2_first5_unlabeled_daeac.npz`
   - `incart_all_daeac.npz`
   - `svdb_all_daeac.npz`

2. Dataset checkpoint MCC, chứa tối thiểu:
   - `daeac_hybrid_mkmmd_mcc_best.pt`
   - optional: `daeac_hybrid_mkmmd_mcc_train_summary.json`

3. Internet bật để `git clone` repo, hoặc thay cell clone bằng copy repo từ Kaggle Dataset nếu repo private.

## 1. Clone repo

Sửa `REPO_URL` và `BRANCH` trước khi chạy. Nếu bạn chạy nhánh khác, đổi `BRANCH` cho đúng commit đã có diagnostics scripts.

In [ ]:
import os, shutil, glob, pathlib, json

REPO_URL = 'https://github.com/your-user/Best-thesis-in-council.git'
BRANCH = 'another-branch'

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'

if not REPO.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    print('Repo already exists:', REPO)

ECG = REPO / 'ecg_thesis'
os.chdir(ECG)
print('cwd =', os.getcwd())

## 2. Install dependencies

Kaggle thường đã có PyTorch, sklearn, matplotlib. Cell này cài requirements repo và `umap-learn` cho embedding plot. Nếu gặp conflict CUDA từ pip, thường vẫn có thể tiếp tục miễn `torch.cuda.is_available()` là `True`.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q umap-learn

import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. Copy preprocessed `.npz` data

Cell này tự tìm folder trong `/kaggle/input` có `mitdb_ds1_daeac.npz`. Nếu Kaggle Dataset của bạn có nhiều bản, có thể set thủ công `DATA_SRC = pathlib.Path('/kaggle/input/.../daeac')`.

In [ ]:
REQUIRED_NPZ = [
    'mitdb_ds1_daeac.npz',
    'mitdb_ds2_daeac.npz',
    'mitdb_ds2_first5_unlabeled_daeac.npz',
    'incart_all_daeac.npz',
    'svdb_all_daeac.npz',
]

data_candidates = [
    p for p in pathlib.Path('/kaggle/input').glob('**')
    if p.is_dir() and all((p / name).exists() for name in REQUIRED_NPZ)
]
assert data_candidates, 'Could not find a Kaggle input folder containing all required DAEAC .npz files.'

DATA_SRC = data_candidates[0]
DATA_DST = ECG / 'data/processed/phase6_daeac_paper'
DATA_DST.mkdir(parents=True, exist_ok=True)

for name in REQUIRED_NPZ:
    shutil.copy2(DATA_SRC / name, DATA_DST / name)

extra = DATA_SRC / 'mitdb_all_daeac.npz'
if extra.exists():
    shutil.copy2(extra, DATA_DST / extra.name)

print('DATA_SRC =', DATA_SRC)
print('Copied files:')
for p in sorted(DATA_DST.glob('*.npz')):
    print(' ', p.name, round(p.stat().st_size / 1024 / 1024, 2), 'MB')

## 4. Copy checkpoint và train summary

Cell này tìm `daeac_hybrid_mkmmd_mcc_best.pt` trong `/kaggle/input`. `train_summary` là optional, nhưng cần để chạy pseudo-label audit đầy đủ.

In [ ]:
METHOD_NAME = 'daeac_hybrid_mkmmd_mcc_best'
EVAL_BATCH_SIZE = 1024
EMBED_METHOD = 'pca'  # use 'umap' for final publication-quality embedding plots; pca is much faster
EMBED_MAX_SAMPLES = 5000
CHECKPOINT_NAME = 'daeac_hybrid_mkmmd_mcc_best.pt'
TRAIN_SUMMARY_NAME = 'daeac_hybrid_mkmmd_mcc_train_summary.json'

ckpt_candidates = glob.glob(f'/kaggle/input/**/{CHECKPOINT_NAME}', recursive=True)
assert ckpt_candidates, f'{CHECKPOINT_NAME} not found under /kaggle/input. Attach checkpoint dataset first.'

CKPT_DST_DIR = ECG / 'outputs/phase6_daeac_hybrid_mkmmd_mcc/checkpoints'
METRIC_DST_DIR = ECG / 'outputs/phase6_daeac_hybrid_mkmmd_mcc/metrics'
CKPT_DST_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DST_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = CKPT_DST_DIR / CHECKPOINT_NAME
shutil.copy2(ckpt_candidates[0], CHECKPOINT)
print('CHECKPOINT =', CHECKPOINT)

summary_candidates = glob.glob(f'/kaggle/input/**/{TRAIN_SUMMARY_NAME}', recursive=True)
TRAIN_SUMMARY = METRIC_DST_DIR / TRAIN_SUMMARY_NAME
if summary_candidates:
    shutil.copy2(summary_candidates[0], TRAIN_SUMMARY)
    print('TRAIN_SUMMARY =', TRAIN_SUMMARY)
else:
    TRAIN_SUMMARY = None
    print('Optional train summary not found. Pseudo-label audit will report unavailable.')

## 5. Static checks

`check_repo.py` kiểm tra cấu trúc repo/config. `00_validate_data.py` kiểm tra shape và label của `.npz` theo config MCC.

In [ ]:
!python scripts/check_repo.py
!python scripts/phase6_daeac_paper/00_validate_data.py --config configs/phase6_daeac_hybrid_mkmmd_mcc.yaml

## 6. Smoke diagnostics trên target 512 mẫu

Chạy cell này trước để bắt lỗi path/checkpoint nhanh. Output smoke sẽ dùng method name `smoke_diag`.

In [ ]:
!python scripts/phase6_daeac_diagnostics/01_collect_predictions.py --config configs/phase6_daeac_diagnostics.yaml --checkpoint {CHECKPOINT} --method-name smoke_diag --dataset target --max-samples 512 --batch-size {EVAL_BATCH_SIZE} --save-embeddings
!python scripts/phase6_daeac_diagnostics/02_failure_tables.py --config configs/phase6_daeac_diagnostics.yaml --method-name smoke_diag --dataset target
!python scripts/phase6_daeac_diagnostics/03_calibration_curves.py --config configs/phase6_daeac_diagnostics.yaml --method-name smoke_diag --dataset target
!python scripts/phase6_daeac_diagnostics/04_pr_roc_analysis.py --config configs/phase6_daeac_diagnostics.yaml --method-name smoke_diag --dataset target
!python scripts/phase6_daeac_diagnostics/05_embedding_analysis.py --config configs/phase6_daeac_diagnostics.yaml --method-name smoke_diag --dataset target --method pca --max-samples 512
!python scripts/phase6_daeac_diagnostics/08_make_diagnostics_report.py --config configs/phase6_daeac_diagnostics.yaml --method-name smoke_diag

## 7. Full collect predictions

Bước này eval checkpoint trên `source_eval`, `target_test`, `INCART`, `SVDB`, rồi lưu predictions và embeddings.

In [ ]:
!python scripts/phase6_daeac_diagnostics/01_collect_predictions.py --config configs/phase6_daeac_diagnostics.yaml --checkpoint {CHECKPOINT} --method-name {METHOD_NAME} --dataset all --batch-size {EVAL_BATCH_SIZE} --save-embeddings

## 8. Full scientific analyses

Các script dưới đây đọc predictions/embeddings đã lưu, tạo CSV, JSON, figures và report.

In [ ]:
!python scripts/phase6_daeac_diagnostics/02_failure_tables.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME}
!python scripts/phase6_daeac_diagnostics/03_calibration_curves.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME}
!python scripts/phase6_daeac_diagnostics/04_pr_roc_analysis.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME}
!python scripts/phase6_daeac_diagnostics/05_embedding_analysis.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME} --method {EMBED_METHOD} --max-samples {EMBED_MAX_SAMPLES}
!python scripts/phase6_daeac_diagnostics/06_morphology_panels.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME} --top-k 24

## 9. Pseudo-label audit và report

Nếu không attach `daeac_hybrid_mkmmd_mcc_train_summary.json`, pseudo-label audit vẫn chạy nhưng sẽ ghi trạng thái unavailable.

In [ ]:
train_summary_arg = '' if TRAIN_SUMMARY is None else f'--train-summary {TRAIN_SUMMARY}'
!python scripts/phase6_daeac_diagnostics/07_pseudo_label_audit.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME} {train_summary_arg}
!python scripts/phase6_daeac_diagnostics/08_make_diagnostics_report.py --config configs/phase6_daeac_diagnostics.yaml --method-name {METHOD_NAME}

## 10. Xem nhanh report và metrics

In [ ]:
REPORT = ECG / f'outputs/phase6_daeac_diagnostics/{METHOD_NAME}_diagnostics_report.md'
print('REPORT =', REPORT)
print(REPORT.read_text(encoding='utf-8')[:6000])

metrics_dir = ECG / 'outputs/phase6_daeac_diagnostics/metrics'
for p in sorted(metrics_dir.glob(f'{METHOD_NAME}_*_metrics.json')):
    data = json.loads(p.read_text(encoding='utf-8'))
    print(p.name, 'accuracy=', round(data.get('accuracy', 0), 4), 'macro_f1=', round(data.get('macro_f1', 0), 4))

## 11. Zip outputs để download

Sau khi cell này chạy xong, download `/kaggle/working/phase6_daeac_diagnostics_outputs.zip` từ panel Output của Kaggle.

In [ ]:
!zip -r /kaggle/working/phase6_daeac_diagnostics_outputs.zip outputs/phase6_daeac_diagnostics
!ls -lh /kaggle/working/phase6_daeac_diagnostics_outputs.zip